In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
from bs4 import BeautifulSoup
import datetime
import pandas as pd
import requests
from time import sleep
import os
import re


In [2]:
# install on the control webiste librbay code. 
# import subprocess
# import sys

# required_packages = ['python-docx',  'pypandoc']

# for package in required_packages:
#     try:
#         __import__(package)
#     except ImportError:
#         print(f'Installing {package}...')
#         subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])


In [3]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'CV BCV' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running CV BCV Web Scraping Tool v.1.1


In [4]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
# No Chrome driver is needed: the BCV page is static and is loaded with requests.get().
driver = None


In [5]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

         regulatorName + ' 1': 'https://www.bcv.cv/pt/Supervisao/Institui%C3%A7%C3%B5es%20Financeiras%20Supervisionadas/Enderecos%20dos%20Bancos/Paginas/EnderecosFAQs.aspx',

        }



Typology={

       regulatorName + ' 1': 'List of Financial Institutions Supervised by BCV',

        }


In [6]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict

LABEL_PATTERNS = {
    'Address_1': [r'Endere[cç]o\s*\.?\s*[:\-]?', r'Endre[cç]o\s*\.?\s*[:\-]?', r'Sede\s*\.?\s*[:\-]?'],
    'Phone': [r'Tel(?:efone|ef\.|\.)?\s*\.?\s*[:\-]?', r'Telefone\s*\.?\s*[:\-]?', r'Telef\s*\.?\s*[:\-]?'],
    'Fax': [r'Fax\s*\.?\s*[:\-]?'],
    'Email': [r'E[-\s]?[Mm]ail\s*\.?\s*[:\-]?', r'Email\s*\.?\s*[:\-]?'],
    'BIC SWIFT Code': [r'Swift\s*\.?\s*[:\-]?'],
    'Website': [r'Site\s*\.?\s*[:\-]?']
}

STOP_LABELS = [pattern for patterns in LABEL_PATTERNS.values() for pattern in patterns] + [
    r'www\.', r'Portal\s*\.?\s*[:\-]?', r'Movel\s*\.?\s*[:\-]?', r'Telm\s*\.?\s*[:\-]?',
    r'Conselho\s+de\s+Administra[cç][aã]o', r'Comiss[aã]o\s+Executiva', r'Administradores\s*:?',
    r'Promotores\s*:?', r'S[oó]cios\s*:?', r'Fiscal\s+[uú]nico', r'Presidente\s*:?',
    r'Data\s+(?:de|da|do)', r'Autorizad[oa]'
]
STOP_PATTERN = r'(?=' + '|'.join(STOP_LABELS) + r'|$)'

def extract_field(text, patterns):
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if not match:
            continue
        value_text = text[match.end():]
        stop = re.search(STOP_PATTERN, value_text, flags=re.IGNORECASE)
        value = value_text[:stop.start()] if stop else value_text
        value = re.sub(r'\s+', ' ', value.replace('\xa0', ' ').replace('\u200b', ' ')).strip(' ;,.-')
        return value
    return ''


def fetch_html(url, timeout=120):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'pt,en-US;q=0.9,en;q=0.8'
    }
    try:
        requests.packages.urllib3.disable_warnings()
    except Exception:
        pass

    response = requests.get(url, headers=headers, timeout=timeout, verify=False)
    response.raise_for_status()
    response.encoding = response.encoding or response.apparent_encoding or 'utf-8'
    return response.text


In [7]:
#------------------------------------------------ Begin_Main ----------------------------------------

for reg, url in regdict.items():
    list_name = Typology.get(reg, 'Unknown List')
    print('Working with {} - {}'.format(reg, list_name))
    
    html = fetch_html(url)
    print('[INFO] : Loaded page with requests.get')
    soup = BeautifulSoup(html, 'html.parser')
    
    headings = []
    for h4 in soup.find_all('h4'):
        name = re.sub(r'\s+', ' ', h4.get_text(' ', strip=True).replace('\xa0', ' ').replace('\u200b', ' ')).strip()
        if name and name.lower() != 'categoria':
            headings.append(h4)

    print(f"[INFO] : Found {len(headings)} entities on {reg}")
    
    for h4 in headings:
        name_val = re.sub(r'\s+', ' ', h4.get_text(' ', strip=True).replace('\xa0', ' ').replace('\u200b', ' ')).strip()
        panel = h4.find_parent('div', class_='panel')
        panel_body = panel.find('div', class_='panel-body') if panel else None
        block_text = panel_body.get_text(' ', strip=True) if panel_body else ''
        block_text = re.sub(r'\s+', ' ', block_text.replace('\xa0', ' ').replace('\u200b', ' ')).strip()
        
        address_val = extract_field(block_text, LABEL_PATTERNS['Address_1'])
        phone_val = extract_field(block_text, LABEL_PATTERNS['Phone'])
        fax_val = extract_field(block_text, LABEL_PATTERNS['Fax'])
        email_match = re.search(r'[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}', block_text)
        email_val = email_match.group(0) if email_match else extract_field(block_text, LABEL_PATTERNS['Email'])
        swift_val = extract_field(block_text, LABEL_PATTERNS['BIC SWIFT Code'])
        website_val = extract_field(block_text, LABEL_PATTERNS['Website'])
        if not website_val:
            website_match = re.search(r'(?:https?://)?www\.[\w\.-]+\.(?:cv|com|net|org)(?:/[^\s]*)?', block_text, flags=re.IGNORECASE)
            website_val = website_match.group(0).strip(' ;,.') if website_match else ''
        
        sqldict['Name'].append(name_val)
        sqldict['Address_1'].append(address_val)
        sqldict['City'].append('Praia' if 'praia' in address_val.lower() else '')
        sqldict['Cntry'].append('CV')
        sqldict['Phone'].append(phone_val)
        sqldict['Fax'].append(fax_val)
        sqldict['Website'].append(website_val)
        sqldict['Email'].append(email_val)
        sqldict['BIC SWIFT Code'].append(swift_val)
        
        sqldict['ListProcessDate'].append(processdate)
        sqldict['RegCtry'].append('CV')
        sqldict['RegCode'].append('BCV')
        sqldict['ListCode'].append(reg.split(' ')[-1])
        sqldict['ListLanguage'].append('Portuguese')
        sqldict['RegulationType'].append('Regulated')
        sqldict['ListName'].append(list_name)

        sqldict = bourange_same_length_array(sqldict)
    
    print(f"[INFO] : Completed {reg}")


Working with CV BCV 1 - List of Financial Institutions Supervised by BCV
[INFO] : Loaded page with requests.get
[INFO] : Found 22 entities on CV BCV 1
[INFO] : Completed CV BCV 1


In [8]:
#------------------------------------------------ Save DataFrame to Excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df.to_excel(filename, 'SQL Ready', index=False)
if driver is not None:
    driver.quit()
sleep(3)
print(f"[INFO] : Excel file '{filename}' saved successfully")


C:\Users\wuj1\AppData\Local\Temp\8\ipykernel_15380\3476817994.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)


[INFO] : Excel file 'CV BCV SQL Ready 2026-05-19 11.16.13.xlsx' saved successfully


In [9]:
#------------------------------------------------ Data Integrity & Consistency Check ----------------------------------------

print("="*80)
print("DATA INTEGRITY & CONSISTENCY VERIFICATION")
print("="*80)

expected_lists = {
    '1': {'name': 'List of Financial Institutions Supervised by BCV', 'expected_rows': 22}
}

print("\n1. DATAFRAME SHAPE:")
print(f"   Total rows collected: {len(df)}")
print(f"   Total columns: {len(df.columns)}")

print("\n2. DATA DISTRIBUTION BY LIST:")
if len(df) > 0:
    list_summary = df.groupby('ListCode').agg({
        'Name': 'count',
        'ListName': 'first',
        'RegCtry': 'first',
        'RegCode': 'first'
    }).rename(columns={'Name': 'Count'})
    print(list_summary)
else:
    print('   NO DATA COLLECTED')

print("\n3. ROW COUNT CHECK:")
actual_rows = len(df)
expected_rows = expected_lists['1']['expected_rows']
row_status = "PASS" if actual_rows == expected_rows else "CHECK"
print(f"   Expected rows: {expected_rows}; actual rows: {actual_rows}; status: {row_status}")

print("\n4. REGCTRY & REGCODE VALIDATION:")
if len(df) > 0:
    regctry_values = df['RegCtry'].unique()
    regcode_values = df['RegCode'].unique()
    language_values = df['ListLanguage'].unique()
    regctry_status = "PASS" if all(v == 'CV' for v in regctry_values) else "FAIL"
    regcode_status = "PASS" if all(v == 'BCV' for v in regcode_values) else "FAIL"
    language_status = "PASS" if all(v == 'Portuguese' for v in language_values) else "FAIL"
    print(f"   RegCtry values: {regctry_values} {regctry_status}")
    print(f"   RegCode values: {regcode_values} {regcode_status}")
    print(f"   ListLanguage values: {language_values} {language_status}")

print("\n5. KEY FIELDS VALIDATION:")
for col in ['Name', 'Address_1', 'Phone', 'Fax', 'Email', 'Website', 'BIC SWIFT Code']:
    filled = (df[col].astype(str).str.strip() != '').sum() if len(df) > 0 else 0
    print(f"   {col} field filled: {filled}/{len(df)}")

print("\n6. SAMPLE DATA (first 5 rows):")
if len(df) > 0:
    print(df[['Name', 'Address_1', 'Phone', 'Fax', 'Email', 'Website', 'BIC SWIFT Code', 'ListCode', 'ListName']].head(5).to_string())
else:
    print('   NO DATA COLLECTED')

print("\n" + "="*80)
print("SUMMARY:")
print("="*80)
print(f"Total rows in DataFrame: {len(df)}")
print(f"Expected list coverage: {df['ListCode'].nunique() if len(df) > 0 else 0}/{len(expected_lists)}")
print("="*80)


DATA INTEGRITY & CONSISTENCY VERIFICATION

1. DATAFRAME SHAPE:
   Total rows collected: 22
   Total columns: 44

2. DATA DISTRIBUTION BY LIST:
          Count                                          ListName RegCtry  \
ListCode                                                                    
1            22  List of Financial Institutions Supervised by BCV      CV   

         RegCode  
ListCode          
1            BCV  

3. ROW COUNT CHECK:
   Expected rows: 22; actual rows: 22; status: PASS

4. REGCTRY & REGCODE VALIDATION:
   RegCtry values: ['CV'] PASS
   RegCode values: ['BCV'] PASS
   ListLanguage values: ['Portuguese'] PASS

5. KEY FIELDS VALIDATION:
   Name field filled: 22/22
   Address_1 field filled: 21/22
   Phone field filled: 21/22
   Fax field filled: 17/22
   Email field filled: 18/22
   Website field filled: 5/22
   BIC SWIFT Code field filled: 3/22

6. SAMPLE DATA (first 5 rows):
                                                                                  

In [10]:

import os
import re
import pandas as pd
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

os.chdir(scriptfolder)
tempfolder = os.path.join(scriptfolder, 'tempfolder')

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

df_pdf = pd.read_excel(filename)

arial_path = r"C:\Windows\Fonts\arial.ttf"
if os.path.exists(arial_path):
    pdfmetrics.registerFont(TTFont('ArialUni', arial_path))
    FONT_NAME = 'ArialUni'
else:
    FONT_NAME = 'Helvetica'
    

def safe_filename(name):
    name = str(name).strip()
    return re.sub(r'[\\/:*?"<>|]+', '_', name) or "UNKNOWN"

def draw_header(c, y):
    c.setFont(FONT_NAME, 14)
    c.drawString(72, y, "Name")
    c.line(72, y - 4, 540, y - 4)
    c.setFont(FONT_NAME, 12)
    return y - 24

def export_list_to_pdf(data_list, pdf_filename):
    c = canvas.Canvas(pdf_filename, pagesize=letter)
    c.setFont(FONT_NAME, 12)

    x = 72
    y = draw_header(c, 740)
    max_lines_per_page = 32
    line_count = 0

    for item in data_list:
        c.drawString(x, y, str(item))
        y -= 20
        line_count += 1

        if line_count >= max_lines_per_page:
            c.showPage()
            c.setFont(FONT_NAME, 12)
            y = draw_header(c, 740)
            line_count = 0

    c.save()

os.makedirs(tempfolder, exist_ok=True)

# One PDF per unique (RegCtry, ListCode) pair so each regulator/list combination
# gets its own file: e.g. "...- CW-1.pdf", "...- SX-1.pdf".
base_name = os.path.splitext(filename)[0]
for (regctry, list_code), group in df_pdf.groupby(['RegCtry', 'ListCode'], dropna=False):
    regctry_safe = safe_filename(regctry)
    list_code_safe = safe_filename(list_code)
    items = group['Name'].dropna().astype(str).tolist()
    if not items:
        continue
    pdf_path = os.path.join(tempfolder, f"{base_name} - {regctry_safe}-{list_code_safe}.pdf")
    export_list_to_pdf(items, pdf_path)
    print(f"[INFO] wrote {len(items)} names -> {os.path.basename(pdf_path)}")


[INFO] wrote 22 names -> CV BCV SQL Ready 2026-05-19 11.16.13 - CV-1.pdf
